**Spark setup**

In [1]:
import os
os.environ['PYSPARK_PYTHON'] = 'python'
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"
os.environ['HADOOP_HOME'] = 'C:\\hadoop' 
from pyspark.sql    import SparkSession
from pyspark        import SparkContext

spark= SparkSession.builder \
    .appName("EcommerceProject") \
    .master('local[*]')\
    .config("spark.driver.memory", "2g")\
    .config('spark.sql.shuffle.partitions', '8')\
    .getOrCreate()

sc= spark.sparkContext

print(f'Spark version  : {spark.version}')
print(f'Python version : {sc.pythonVer}')
print(f'App name       : {sc.appName}')
print(f'Master         : {sc.master}')
print(f'Cores available: {sc.defaultParallelism}')
print()
print('Spark UI → http://localhost:4040')
    

Spark version  : 4.1.1
Python version : 3.11
App name       : EcommerceProject
Master         : local[*]
Cores available: 12

Spark UI → http://localhost:4040


Reading csv file 

In [2]:
rawdf=(
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("nullValue", "")
    .option('multiLine', 'true')
    .option('escape', '"')
    .csv('..\\data\\ecommerce_logs.csv')
)
print(f'Shape of the raw dataframe: {rawdf.count()} rows x {len(rawdf.columns)} columns')
print()
#rawdf.show(10,truncate=False)

Shape of the raw dataframe: 10831817 rows x 9 columns



In [3]:
rawdf.printSchema()
print()


root
 |-- timestamp: timestamp (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- price: double (nullable = true)
 |-- referrer: string (nullable = true)
 |-- user_metadata: string (nullable = true)
 |-- product_metadata: string (nullable = true)




Inspecting the metdata 
and extracting categories as well 

In [8]:
rawdf.select(
"user_metadata",
    "product_metadata"
).show(5, truncate=False)

+-------------------------------------------------+------------------------------------------------------------+
|user_metadata                                    |product_metadata                                            |
+-------------------------------------------------+------------------------------------------------------------+
|{"device": "Mobile", "tier": "Gold", "loc": "US"}|{"category": "Books", "brand": "HarperCollins", "stock": 83}|
|{"device": "Mobile", "tier": "Gold", "loc": "US"}|{"category": "Electronics", "brand": "Samsung", "stock": 67}|
|{"device": "Mobile", "tier": "Gold", "loc": "US"}|{"category": "Electronics", "brand": "Dell", "stock": 70}   |
|NULL                                             |NULL                                                        |
|NULL                                             |{"category": "Clothing", "brand": "H&M", "stock": 55}       |
+-------------------------------------------------+---------------------------------------------

In [6]:
import pyspark.sql.functions as F

print('Null counts per column:')
null_counts = rawdf.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in rawdf.columns
])
null_counts.show()

Null counts per column:


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "c:\Users\Mira\spark_env\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Mira\spark_env\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Mira\AppData\Local\Programs\Python\Python311\Lib\socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [7]:
import pyspark.sql.functions as F

#print("Original row count:", rawdf.count())

# 1. Extract JSON fields into their own columns
parsed_df = rawdf.withColumn(
    "category", F.get_json_object(F.col("product_metadata"), "$.category")
).withColumn(
    "brand", F.get_json_object(F.col("product_metadata"), "$.brand")
).withColumn(
    "device", F.get_json_object(F.col("user_metadata"), "$.device")
)

# 2. Data Cleaning
cleaned_df = (
    parsed_df
    # Handle Null Values & Missing Categories (Keep rows where critical fields are NOT null)
    .dropna(subset=["session_id", "user_id", "product_id", "event_type", "timestamp", "category","user_metadata"])
    # Handle Incorrect Timestamps & Invalid Records (e.g., price must be > 0)
    .filter(F.col("timestamp").isNotNull())
    .filter(F.col("price") > 0)
)
print("dropped row count:", rawdf.count()-cleaned_df.count())


# Show the new schema and a few rows
cleaned_df.show(5, truncate=False)



dropped row count: 3724700
+-------------------+---------------+---------+----------+----------+------+-------------+-------------------------------------------------+------------------------------------------------------------+-----------+-------------+------+
|timestamp          |session_id     |user_id  |event_type|product_id|price |referrer     |user_metadata                                    |product_metadata                                            |category   |brand        |device|
+-------------------+---------------+---------+----------+----------+------+-------------+-------------------------------------------------+------------------------------------------------------------+-----------+-------------+------+
|2026-02-21 08:33:36|SESS_4007a38849|User_2686|view      |ITEM_3306 |754.5 |instagram.com|{"device": "Mobile", "tier": "Gold", "loc": "US"}|{"category": "Books", "brand": "HarperCollins", "stock": 83}|Books      |HarperCollins|Mobile|
|2026-02-21 08:34:02|SESS_4007a38

In [8]:
df=cleaned_df
cleaned_df.select("event_type").distinct().show()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "c:\Users\Mira\spark_env\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Mira\spark_env\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Mira\AppData\Local\Programs\Python\Python311\Lib\socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
ERROR:py4j.clientserver:Exception occurred while shutting down connection
Traceback (most recent call last):
  File "c:\Users\Mira\spark_env\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Mira

KeyboardInterrupt: 

Lets start basket Analysis phase 1.1

In [9]:
# 4.4 Event Separation
views_df = cleaned_df.filter(F.col("event_type") == "view")
cart_df = cleaned_df.filter(F.col("event_type") == "cart")
purchase_df = cleaned_df.filter(F.col("event_type") == "purchase")

total   =views_df.count()+cart_df.count()+ purchase_df.count()
# Let's verify how many rows we have for each event!
print("Total views:", views_df.count())
print("Total carts:", cart_df.count())
print("Total purchases:", purchase_df.count())
print("Total Count:",total ,total==cleaned_df.count() )


Total views: 5330958
Total carts: 1421221
Total purchases: 354938
Total Count: 7107117 True


Checking partions

In [12]:
print(f'Default partitions: {cleaned_df.rdd.getNumPartitions()}')

# spark.sql.shuffle.partitions controls post-shuffle partitions (groupBy, join)
print(f'Shuffle partitions: {spark.conf.get("spark.sql.shuffle.partitions")}')

# For small data, 200 (default) is wasteful — we set 8 at startup
# For large data on a real cluster: num_cores × 2-4 is a good starting point

# Count records per partition
print('\nRecords per partition:')
part_sizes = cleaned_df.rdd.mapPartitionsWithIndex(
    lambda idx, it: [(idx, sum(1 for _ in it))]
).collect()
for pidx, cnt in part_sizes:
    bar = '#' * (cnt // 10)
    print(f'  Partition {pidx}: {cnt:>4} rows  {bar}')

Default partitions: 1
Shuffle partitions: 8

Records per partition:
  Partition 0: 7107117 rows  #######################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################

In [10]:
# 4.5 Performance Optimization

# Use 48 partitions (12 cores * 4) for optimal parallel processing
num_partitions = 48

# 1. Optimize the main dataframe 
cleaned_df = cleaned_df.repartition(num_partitions)
cleaned_df.cache()

# 2. Optimize YOUR specific dataframe for Market Basket Analysis (Partitioned specifically by session)
optimized_purchase_df = purchase_df.repartition(num_partitions, "session_id")
optimized_purchase_df.cache()

# Force Spark to execute the caching into memory
print("Total clean events cached :", cleaned_df.count())

Total clean events cached : 7107117


*Map*

In [12]:
import itertools
purchase_rdd = optimized_purchase_df.select("session_id", "product_id").rdd
# MAP STEP: Emit (session_id, item_id)
mapped_rdd = purchase_rdd.map(lambda x: (x.session_id, x.product_id))

#REDUCE STEP 1: Group by Session
# Group by key (session_id) and convert the items into list
grouped_rdd = mapped_rdd.groupByKey().mapValues(lambda items: list(set(items)))

# 4.7 / 4.8 REDUCE STEP 2: Generate Product Pairs
def generate_pairs(items):
    # Sort items so (Item A, Item B) is the same as (Item B, Item A)
    sorted_items = sorted(items)
    # Generate all combinations of length 2
    return list(itertools.combinations(sorted_items, 2))

# flatMap extracts the pairs out of the session groups into a flat list of pairs
pairs_rdd = grouped_rdd.flatMap(lambda x: generate_pairs(x[1]))





In [ ]:
# ==========================================
# 4.9 Recommendation Pair Counting
# Classic WordCount logic: map each pair to (pair, 1), then reduceByKey to add them up
pair_frequencies_rdd = pairs_rdd.map(lambda pair: (pair, 1)).reduceByKey(lambda a, b: a + b)

# Sort by the most frequently bought together pairs and print the Top 10!
top_recommendations = pair_frequencies_rdd.sortBy(lambda x: x[1], ascending=False)

print("Top 10 Most Frequently Bought Together Product Pairs:")
for pair, count in top_recommendations.take(10):
    print(f"Pair: {pair} | Bought Together: {count} times")

In [ ]:
# ==========================================
# 1. Save the Cleaned Dataset
# ==========================================
print("Saving cleaned dataset as Parquet...")
# We use 'overwrite' so you don't get an error if you run this cell twice!
cleaned_df.write.mode("overwrite").csv("../data/cleaned_ecommerce_logs.csv")
print("Cleaned data saved successfully!")

# ==========================================
# 2. Save the Market Basket Recommendations
# ==========================================
print("Saving Market Basket recommendations...")

# First, our RDD looks like: (('PS5', 'Controller'), 3200)
# We map it to flatten the pair so it looks like: ('PS5', 'Controller', 3200)
flat_pairs_rdd = pair_frequencies_rdd.map(lambda x: (x[0][0], x[0][1], x[1]))

# Convert the flattened RDD back into a beautiful PySpark DataFrame
recommendations_df = flat_pairs_rdd.toDF(["item_1", "item_2", "frequency"])

# Save it as a standard CSV with headers
recommendations_df.write.mode("overwrite").csv("../data/market_basket_recommendations.csv", header=True)
print("Recommendations saved successfully!")


Saving cleaned dataset as Parquet...


Py4JJavaError: An error occurred while calling o321.csv.
: java.lang.RuntimeException: java.io.FileNotFoundException: Hadoop bin directory does not exist: C:\hadoop\bin\bin -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.ja